# 경매 220건에 공시지가·건축물대장 붙이기 (VS Code 주피터)

**준비 (처음 한 번)**
1. 이 노트북과 **같은 폴더**(또는 그 아래 `data` 폴더)에 `경매_매각사례_시험셋.xlsx`를 둬요. 드라이브 `신보프로젝트/data/`에서 내려받으면 돼요.
2. VS Code 오른쪽 위 **커널 선택**에서 파이썬을 고르고, 아래 '패키지 설치' 칸을 한 번 실행해요.
3. 인증키 두 개: V-World 키, 공공데이터포털 키(온비드 때 쓴 키). 0단계를 실행하면 위쪽에 입력창이 떠요.

**실행**: 위에서부터 칸마다 ▶ (또는 Shift+Enter). 끝나면 생긴 `auction_features.csv`를 드라이브 `신보프로젝트/data/`에 올리고, 코랩에서 `auction_calibration_colab.ipynb`를 다시 '모두 실행'하세요.

## 패키지 설치 (처음 한 번)

In [ ]:
%pip install -q requests pandas openpyxl

## 0단계. 설정 + 인증키

In [ ]:
import os, re, json, time, glob, getpass
import numpy as np
import pandas as pd
import requests

try:
    BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE = os.getcwd()
# 이 폴더 → data 폴더 → 다운로드 폴더에서 이름에 '시험셋'이 들어간 가장 최근 엑셀
_c = []
for _d in [BASE, os.path.join(BASE, "data"), os.path.join(os.path.expanduser("~"), "Downloads")]:
    _c += glob.glob(os.path.join(_d, "*시험셋*.xlsx"))
_c = [c for c in _c if not os.path.basename(c).startswith("~$")]          # 엑셀 임시파일 제외
XLSX = max(_c, key=os.path.getmtime) if _c else None
OUT_DIR = os.path.dirname(XLSX) if XLSX else BASE
CACHE = os.path.join(OUT_DIR, "auction_feat_cache.jsonl")       # 중간 저장 (다시 실행하면 이어서)
OUT = os.path.join(OUT_DIR, "auction_features.csv")

VW_KEY = os.environ.get("VWORLD_KEY", "").strip() or getpass.getpass("V-World 인증키: ").strip()
GO_KEY = os.environ.get("DATA_GO_KR_KEY", "").strip() or os.environ.get("ONBID_KEY", "").strip() \
         or getpass.getpass("공공데이터포털 인증키(Decoding): ").strip()

URL_GEOCODE = "https://api.vworld.kr/req/address"
URL_DATA    = "https://api.vworld.kr/req/data"
URL_PRICE   = "https://api.vworld.kr/ned/data/getIndvdLandPriceAttr"
URL_BLD     = "https://apis.data.go.kr/1613000/BldRgstHubService/getBrTitleInfo"

print("입력 파일:", XLSX or "없음 ⚠️ (경매_매각사례_시험셋.xlsx 를 이 폴더에 두세요)")
print("결과 파일:", OUT)

## 1단계. 220건 읽기 + 주소 정리
'외 1필지', '○○타워 3층', '(平沙里)' 같은 꼬리를 떼고 지번까지만 남겨요.

In [ ]:
cases = pd.read_excel(XLSX, sheet_name="② 사례 기록", header=4)
cases = cases[cases["★ 사건번호"].notna()].copy()
cases["사건"] = (cases["★ 법원"].astype(str) + " " + cases["★ 사건번호"].astype(str) + "-"
                + cases["★ 물건번호"].astype(int).astype(str))            # 보정 노트북과 같은 키

JIBUN = re.compile(r"^(.*?(?:동|리|가|로|길)\s*(?:산\s?)?\d+(?:-\d+)?)")
def clean_addr(a):
    """'…동 643-7 외 1필지', '…동 207-5외 1필지, ○○타워 3층', '…리(平沙里) 838', '…동 610-1옥계오션월드' → 지번까지만"""
    s = re.sub(r"\(.*?\)", "", str(a)).split(",")[0].strip()
    m = JIBUN.match(s)
    return re.sub(r"\s+", " ", m.group(1)).strip() if m else s

cases["addr"] = cases["★ 소재지 (지번까지)"].map(clean_addr)
cases["sale_date"] = pd.to_datetime(cases["★ 매각기일"], errors="coerce")
print(len(cases), "건")
print(cases[["사건", "★ 소재지 (지번까지)", "addr"]].head(8).to_string(index=False))

## 2단계. 조회 함수 + 시험 1건
마지막 줄에 PNU와 공시지가 연도, `건축물: True/False`가 나오면 키가 정상이에요.

In [ ]:
def get_json(url, params):
    for i in range(3):
        try:
            r = requests.get(url, params=params, timeout=20)
            return r.json()
        except Exception as e:
            if i == 2:
                print("  ! 호출 실패:", url.split("/")[-1], e)
            time.sleep(1 + i)
    return None

def find_rows(js, field):
    if isinstance(js, dict):
        if field in js: return [js]
        for v in js.values():
            r = find_rows(v, field)
            if r: return r
    elif isinstance(js, list):
        hit = [x for x in js if isinstance(x, dict) and field in x]
        if hit: return hit
        for v in js:
            r = find_rows(v, field)
            if r: return r
    return []

def addr_to_pnu(addr):
    """V-World 지오코더 → 19자리 PNU (표준: 11번째 자리 1=일반, 2=산). 결과 지번이 다르면 표시"""
    g = get_json(URL_GEOCODE, {"key": VW_KEY, "format": "json", "service": "address", "request": "getcoord",
                               "version": "2.0", "crs": "epsg:4326", "type": "parcel", "refine": "true",
                               "simple": "false", "address": addr})
    try:
        resp = g["response"]
        if resp["status"] != "OK":
            return None, f"geocode_{resp['status']}", None
        st = resp.get("refined", {}).get("structure", {})
        pnu = st.get("level4LC")
        matched = resp.get("refined", {}).get("text")
        if not (pnu and len(str(pnu)) == 19):                     # level4LC 가 없으면 지적도로
            x, y = resp["result"]["point"]["x"], resp["result"]["point"]["y"]
            d = get_json(URL_DATA, {"key": VW_KEY, "format": "json", "service": "data", "request": "GetFeature",
                                    "data": "LP_PA_CBND_BUBUN", "geomFilter": f"POINT({x} {y})", "crs": "EPSG:4326",
                                    "size": 1, "geometry": "false", "attribute": "true"})
            rows = find_rows(d, "pnu")
            pnu = rows[0]["pnu"] if rows else None
        return (str(pnu) if pnu else None), ("ok" if pnu else "parcel_fail"), matched
    except Exception:
        return None, "geocode_fail", None

def land_prices(pnu):
    """{기준연도: ㎡당 공시지가}"""
    rows = find_rows(get_json(URL_PRICE, {"key": VW_KEY, "format": "json", "pnu": pnu,
                                          "numOfRows": 100, "pageNo": 1}), "pblntfPclnd")
    out = {}
    for r in rows:
        try:
            out[int(r["stdrYear"])] = float(str(r["pblntfPclnd"]).replace(",", ""))
        except Exception:
            pass
    return out

def building(pnu):
    """건축물대장 표제부 — 주건축물 중 연면적 최대 동"""
    plat = {"1": "0", "2": "1"}.get(pnu[10], "0")                  # 표준 PNU 1=일반→0, 2=산→1
    js = get_json(URL_BLD, {"serviceKey": GO_KEY, "sigunguCd": pnu[0:5], "bjdongCd": pnu[5:10],
                            "platGbCd": plat, "bun": pnu[11:15], "ji": pnu[15:19],
                            "numOfRows": 100, "pageNo": 1, "_type": "json"})
    rows = find_rows(js, "mainPurpsCdNm")
    if not rows:
        return {"bld_found": False}
    main = [r for r in rows if "주" in str(r.get("mainAtchGbCdNm", "주"))] or rows
    def num(v):
        x = pd.to_numeric(str(v).replace(",", ""), errors="coerce")
        return float(x) if pd.notna(x) else None
    best = max(main, key=lambda r: num(r.get("totArea")) or -1)
    return {"bld_found": True, "bld_n_dong": len(main), "bld_use_apr": str(best.get("useAprDay") or "").strip(),
            "bld_strct": best.get("strctCdNm"), "bld_purps": best.get("mainPurpsCdNm"),
            "bld_grnd_flr": num(best.get("grndFlrCnt")), "bld_ugrnd_flr": num(best.get("ugrndFlrCnt")),
            "bld_tot_area": num(best.get("totArea")), "bld_regstr_gb": best.get("regstrGbCdNm")}

# 키 확인용: 1건만 미리 호출
t_pnu, t_st, t_m = addr_to_pnu(cases["addr"].iloc[0])
print("시험:", cases["addr"].iloc[0], "→", t_pnu, t_st, "| 매칭:", t_m)
if t_pnu:
    print("  공시지가 연도:", sorted(land_prices(t_pnu))[-3:], "| 건축물:", building(t_pnu).get("bld_found"))

## 3단계. 220건 조회
몇 분 걸려요. 끊기면 다시 실행하면 이어서 해요 (`auction_feat_cache.jsonl`).

In [ ]:
done = {}
if os.path.exists(CACHE):
    for line in open(CACHE, encoding="utf-8"):
        if line.strip():
            c = json.loads(line); done[c["addr"]] = c

todo = [a for a in cases["addr"].unique() if a not in done]
print(f"주소 {cases['addr'].nunique()}개 중 새로 조회 {len(todo)}개")
with open(CACHE, "a", encoding="utf-8") as f:
    for i, a in enumerate(todo, 1):
        pnu, st, matched = addr_to_pnu(a)
        rec = {"addr": a, "pnu": pnu, "status": st, "matched": matched, "prices": {}, "bld": {"bld_found": False}}
        if pnu:
            rec["prices"] = land_prices(pnu)
            rec["bld"] = building(pnu)
        done[a] = rec
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n"); f.flush()
        if i % 20 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)}")
        time.sleep(0.1)

## 4단계. 피처 만들기 → `auction_features.csv` 저장

In [ ]:
def pick_price(prices, d):
    if not prices or pd.isna(d): return np.nan, np.nan
    limit = d.year if d.month >= 6 else d.year - 1                 # 1~5월 매각 → 전년도 (발표 시점 누수 방지)
    ok = [int(y) for y in prices if int(y) <= limit]
    if not ok: return np.nan, np.nan
    return prices[str(max(ok))], max(ok)

rows = []
for _, c in cases.iterrows():
    rec = done.get(c["addr"], {})
    prices = {str(k): v for k, v in (rec.get("prices") or {}).items()}
    lp, yr = pick_price(prices, c["sale_date"])
    land = pd.to_numeric(c["토지면적(㎡)"], errors="coerce")
    apsl = pd.to_numeric(c["★ 감정가(원)"], errors="coerce")
    b = rec.get("bld") or {}
    age = np.nan
    ua = str(b.get("bld_use_apr", "") or "")
    if b.get("bld_found") and re.fullmatch(r"\d{8}", ua):
        if ua[4:6] == "00": ua = ua[:4] + "0701"          # 월이 00 → 그해 중간(7/1)으로
        elif ua[6:8] == "00": ua = ua[:6] + "15"          # 일이 00 (예: 19350600) → 그달 15일로
        d0 = pd.to_datetime(ua, format="%Y%m%d", errors="coerce")
        if pd.notna(d0):
            age = (c["sale_date"] - d0).days / 365.25
    # 토지 물건(건물면적 없음)은 대장에 건물이 잡혀도 건물 피처를 비움 (학습 때 토지와 같은 모양)
    is_land = pd.isna(pd.to_numeric(c["건물면적(㎡)"], errors="coerce"))
    rows.append({
        "사건": c["사건"], "addr_clean": c["addr"], "pnu": rec.get("pnu"), "vw_status": rec.get("status"),
        "vw_matched": rec.get("matched"),
        "지번일치": (str(rec.get("matched") or "").split()[-1:] == str(c["addr"]).split()[-1:]),
        "land_price": lp, "price_year": yr,
        "log_land_price": np.log(lp) if pd.notna(lp) and lp > 0 else np.nan,
        "apsl_to_official": apsl / (lp * land) if pd.notna(lp) and pd.notna(land) and lp * land > 0 else np.nan,
        "bld_found": bool(b.get("bld_found")) and not is_land,
        "bld_age_c": np.clip(age, 0, 80) if not is_land else np.nan,
        "bld_grnd_flr": b.get("bld_grnd_flr") if not is_land else np.nan,
        "bld_tot_area": b.get("bld_tot_area") if not is_land else np.nan,
        "bld_strct": b.get("bld_strct") if not is_land else np.nan,
        "bld_purps": b.get("bld_purps") if not is_land else np.nan,
    })
feat = pd.DataFrame(rows)
feat.to_csv(OUT, index=False, encoding="utf-8-sig")

print("저장:", OUT)
print("\n조회 상태:"); print(feat["vw_status"].value_counts(dropna=False).to_string())
print(f"\n공시지가 붙음: {feat['land_price'].notna().sum()} / {len(feat)}")
print(f"건축물대장 붙음(건물 물건): {feat['bld_found'].sum()} / {(~cases['건물면적(㎡)'].isna()).sum()}")
print(f"지오코더가 다른 지번으로 바꾼 건: {(~feat['지번일치']).sum()}건 (아래 확인)")
print(feat.loc[~feat["지번일치"], ["사건", "addr_clean", "vw_matched"]].head(15).to_string(index=False))
print("\n→ auction_features.csv 를 드라이브 신보프로젝트/data/ 에 올리고, 보정 노트북을 다시 '모두 실행'하세요.")